In [62]:
import pandas as pd
import numpy as np
from scipy.interpolate import griddata
import tifffile as tf

original_data_path = "../data/raw/"
data_binary_representation ="../data/interim/"

In [3]:
df = pd.read_csv(original_data_path + "wells.csv")
df_concentration = pd.read_csv(original_data_path + "concentration.csv")
df_head = pd.read_csv(original_data_path + "head.csv")
df_cores = pd.read_csv(original_data_path + "cores.csv")
df_pumping = pd.read_csv(original_data_path + "pumping.csv")

Сначала удалим все nan значение, затем удалим каскадно из других таблиц, после удалим дубликаты с округлением по x и y до целых и восстановим z.

In [4]:
df.loc[df['x'] == -999.0, "x"] = np.nan

nan_values_mask = df[['x','y','z']].isna().any(axis=1)

df_deleted = df[nan_values_mask]
df_deleted = df_deleted["well_id"].unique()
df = df[~nan_values_mask]

df["round_x"] = df["x"].round(0)
df["round_y"] = df["y"].round(0)

duplicated_mask = df.duplicated(subset=["round_x", "round_y", "z"], keep="first")
duplicated_idx = df[duplicated_mask]["well_id"].unique()

df = df[~duplicated_mask]
df = df.drop(columns=["round_x", "round_y"])

In [5]:
df_concentration = df_concentration[~df_concentration["well_id"].isin(df_deleted)]
df_head = df_head[~df_head["well_id"].isin(df_deleted)]
df_cores = df_cores[~df_cores["well_id"].isin(df_deleted)]
df_pumping = df_pumping[~df_pumping["well_id"].isin(df_deleted)]

df_concentration = df_concentration[~df_concentration["well_id"].isin(duplicated_idx)]
df_head = df_head[~df_head["well_id"].isin(duplicated_idx)]
df_cores = df_cores[~df_cores["well_id"].isin(duplicated_idx)]
df_pumping = df_pumping[~df_pumping["well_id"].isin(duplicated_idx)]

In [6]:
df.loc[(df["z"] > 5) & (df["z"] < 150), "z"] = -df["z"]

In [7]:
df.to_parquet(f"{data_binary_representation}wells.parquet", engine='pyarrow')

In [8]:
df_cores.loc[df_cores["z_top"] > 0, "z_top"] = -df_cores["z_top"]

df_cores = df_cores.sort_values(by=["well_id", "z_top"], ascending=[True, False])
df_cores["z_bottom"] = df_cores["z_bottom"].fillna(df_cores.groupby("well_id")["z_top"].shift(-1))

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom", "lithology"], na_position="last")
df_cores = df_cores.drop_duplicates(subset=["well_id", "z_top", "z_bottom"], keep="first")

df_cores.loc[df_cores["lithology"].isin(["Clay", "Caly"]), "lithology"] = "clay"
df_cores.loc[df_cores["lithology"].isin(["Gravel"]), "lithology"] = "gravel"
df_cores.loc[df_cores["lithology"].isin(["Loam"]), "lithology"] = "loam"
df_cores.loc[df_cores["lithology"].isin(["Sand", "SAND", "sand "]), "lithology"] = "sand"

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom"], ascending=[True, False, True])
prev_max_deph = df_cores.groupby("well_id")["z_bottom"].cummin().shift(1)
same_well = df_cores["well_id"] == df_cores["well_id"].shift(1)
is_inner = same_well & (df_cores["z_bottom"] >= prev_max_deph)
df_cores = df_cores[~is_inner]

df["round_x"] = df["x"].round(0)
df["round_y"] = df["y"].round(0)
duplicated_wells = df.duplicated(subset=["round_x", "round_y"], keep=False)
wells_names = df[duplicated_wells].copy()

depths = df_cores.groupby("well_id")["z_bottom"].min().reset_index()
merged_df = wells_names.merge(depths, on="well_id")
trust_index = merged_df.groupby(by=["round_x", "round_y"])["z_bottom"].idxmin()
trusted_wells = merged_df.loc[trust_index, "well_id"]

bad_wells = set(wells_names["well_id"]) - set(trusted_wells)
df_cores = df_cores[~df_cores["well_id"].isin(bad_wells)]

df_cores = df_cores.sort_values(by=["well_id", "z_top", "z_bottom"], ascending=[True, False, True])
prev_z_bottom = df_cores.groupby("well_id")["z_bottom"].shift(1)
df_cores["z_top"] = np.minimum(df_cores["z_top"], prev_z_bottom.fillna(df_cores["z_top"]))

df_cores = df_cores.drop_duplicates(subset=["well_id", "z_top", "z_bottom", "lithology"], keep="first")

df_cores = df_cores.dropna(subset=["z_bottom", "lithology"])

In [9]:
df_cores.to_parquet(f"{data_binary_representation}cores.parquet")

In [10]:
df_head = df_head[df_head["head"] >= 0]

df_head = df_head.dropna(subset=["t", "head"])
df_head = df_head.drop_duplicates(subset=["well_id", "t"], keep="first")

df_head["t_datetime"] = pd.to_datetime(df_head["t"], format="%d.%m.%Y", errors="coerce")
bad_values = (
    (df_head["t_datetime"].isna()) |
    (df_head["t_datetime"] < "2020-01-01") |
    (df_head["t_datetime"] > "2025-10-31")
)
df_head = df_head[~bad_values]

df_head = df_head.drop(columns=["t"])
df_head = df_head.rename(columns={"t_datetime": "t"})

In [11]:
df_head.to_parquet(f"{data_binary_representation}head.parquet")

In [12]:
df_pumping.loc[df_pumping["units"] == "l/s", "pumping_rate"] = (df_pumping["pumping_rate"] * 24 * 60 * 60 / 1000)
df_pumping.loc[df_pumping["units"] == "l/s", "units"] = "m3/day"

df_pumping = df_pumping.dropna(subset=["pumping_rate"])

df_pumping["t_datetime"] = pd.to_datetime(df_pumping["t"], format="%d.%m.%Y", errors="coerce")
bad_values = (
    (df_pumping["t_datetime"].isna()) |
    (df_pumping["t_datetime"] < "2020-01-01") |
    (df_pumping["t_datetime"] > "2025-12-01")
)
df_pumping = df_pumping[~bad_values]

df_pumping = df_pumping.drop(columns=["t"])
df_pumping = df_pumping.rename(columns={"t_datetime":"t"})

df_pumping.loc[df_pumping["pumping_rate"] < 0, "pumping_rate"] = -df_pumping["pumping_rate"]

In [21]:
df_pumping.to_parquet(f"{data_binary_representation}pumping.parquet", engine="pyarrow")

In [22]:
df_concentration.loc[df_concentration["concentration"].isin(["ND", "<0.05"]), "concentration"] = "0.0"

df_concentration["concentration"] = pd.to_numeric(df_concentration["concentration"], errors="coerce")

df_concentration["t"] = pd.to_datetime(df_concentration["t"], format="%d.%m.%Y", errors="coerce")

bad_values = df_concentration["t"].isna()

df_concentration = df_concentration[~bad_values]

df_concentration = df_concentration[df_concentration["concentration"] >= 0]

In [27]:
df_concentration.to_parquet(f"{data_binary_representation}concentration.parquet", engine="pyarrow")

In [28]:
df_abstraction = pd.read_csv(f"{original_data_path}abstraction.csv")

In [29]:
df_abstraction = df_abstraction.dropna(subset=["total_abstraction_m3_day"])

df_abstraction["t_datetime"] = pd.to_datetime(df_abstraction["t"], format="%d.%m.%Y", errors="coerce")

bad_values = (
    (df_abstraction["t_datetime"].isna()) |
    (df_abstraction["t_datetime"] < "2020-01-01") |
    (df_abstraction["t_datetime"] > "2025-12-01")
)

df_abstraction["t"] = pd.to_datetime(df_abstraction["t"], format="%d.%m.%Y", errors="coerce")

df_abstraction.drop(columns=["t_datetime"], inplace=True)

df_abstraction = df_abstraction[~bad_values]

In [39]:
df_abstraction.to_parquet(f"{data_binary_representation}abstraction.parquet", engine="pyarrow")

In [40]:
df_queries_test = pd.read_csv(f"{original_data_path}queries_test.csv")

In [43]:
df_queries_test["t"] = pd.to_datetime(df_queries_test["t"], format="%d.%m.%Y", errors="coerce")
display(df_queries_test)
display(df_queries_test.describe())

,query_id,x,y,z,t
0,Q_0001,1444.29,924.32,-86.98,2024-06-28
1,Q_0002,3013.90,2212.32,-38.01,2022-07-30
2,Q_0003,3180.09,3104.40,-52.31,2025-07-27
3,Q_0004,3371.66,2417.84,-87.11,2026-02-20
4,Q_0005,1646.35,1536.63,-83.42,2024-08-06
...,...,...,...,...,...
595,Q_0596,3816.33,3491.77,-32.24,2024-05-24
596,Q_0597,2169.66,1421.34,-54.87,2022-06-11
597,Q_0598,1711.11,2293.67,-116.73,2024-02-15
598,Q_0599,2510.13,2214.94,-29.55,2025-10-06


,x,y,z,t
count,600.000000,600.000000,600.000000,600
mean,2744.489483,2456.737333,-65.805500,2024-03-30 20:31:12
min,265.060000,189.350000,-119.550000,2022-01-05 00:00:00
25%,1929.082500,1782.907500,-94.897500,2023-03-09 12:00:00
50%,2674.600000,2450.195000,-65.155000,2024-02-19 00:00:00
75%,3554.695000,3167.767500,-38.370000,2025-05-16 00:00:00
max,5000.000000,4794.660000,-10.140000,2026-06-30 00:00:00
std,1035.599365,910.226114,31.910898,NaN


In [44]:
df_queries_test.to_parquet(f"{data_binary_representation}queries_test.parquet", engine="pyarrow")

In [45]:
df_ground_truth_answers = pd.read_csv(f"{original_data_path}_ground_truth_answers.csv")

In [47]:
df_ground_truth_answers["t"] = pd.to_datetime(df_ground_truth_answers["t"], format="%d.%m.%Y", errors="coerce")
display(df_ground_truth_answers)
display(df_ground_truth_answers.describe())
display(df_ground_truth_answers.info())

,query_id,x,y,z,t,true_concentration
0,Q_0001,1444.29,924.32,-86.98,2024-06-28,14.85
1,Q_0002,3013.90,2212.32,-38.01,2022-07-30,11.54
2,Q_0003,3180.09,3104.40,-52.31,2025-07-27,302.54
3,Q_0004,3371.66,2417.84,-87.11,2026-02-20,60.90
4,Q_0005,1646.35,1536.63,-83.42,2024-08-06,50.10
...,...,...,...,...,...,...
595,Q_0596,3816.33,3491.77,-32.24,2024-05-24,6.05
596,Q_0597,2169.66,1421.34,-54.87,2022-06-11,110.42
597,Q_0598,1711.11,2293.67,-116.73,2024-02-15,1.33
598,Q_0599,2510.13,2214.94,-29.55,2025-10-06,428.75


,x,y,z,t,true_concentration
count,600.000000,600.000000,600.000000,600,600.00000
mean,2744.489483,2456.737333,-65.805500,2024-03-30 20:31:12,80.40525
min,265.060000,189.350000,-119.550000,2022-01-05 00:00:00,0.10000
25%,1929.082500,1782.907500,-94.897500,2023-03-09 12:00:00,0.71750
50%,2674.600000,2450.195000,-65.155000,2024-02-19 00:00:00,13.56500
75%,3554.695000,3167.767500,-38.370000,2025-05-16 00:00:00,94.30750
max,5000.000000,4794.660000,-10.140000,2026-06-30 00:00:00,1143.32000
std,1035.599365,910.226114,31.910898,NaN,144.70849


<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   query_id            600 non-null    str           
 1   x                   600 non-null    float64       
 2   y                   600 non-null    float64       
 3   z                   600 non-null    float64       
 4   t                   600 non-null    datetime64[us]
 5   true_concentration  600 non-null    float64       
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 31.8 KB


None

In [48]:
df_ground_truth_answers.to_parquet(f"{data_binary_representation}_ground_trust_answers.parquet", engine="pyarrow")

In [49]:
df_recharge = pd.read_csv(f"{original_data_path}recharge.csv")

In [50]:
df_recharge.dropna(subset=["recharge_mm"], inplace=True)
df_recharge["t"] = pd.to_datetime(df_recharge["t"], format="%d.%m.%Y", errors="coerce")
error_values = df_recharge["recharge_mm"] < 0

df_recharge = df_recharge[~error_values]

In [52]:
df_recharge["t"] = pd.to_datetime(df_recharge["t"], format="%d.%m.%Y", errors="coerce")
display(df_recharge)
display(df_recharge.describe())
display(df_recharge.info())

,zone,t,recharge_mm
0,Industrial_Zone,2020-01-01,0.65
1,Water_Intake_Zone,2020-01-01,0.00
2,Forest_Zone,2020-01-01,1.89
3,Industrial_Zone,2020-02-01,4.65
4,Water_Intake_Zone,2020-02-01,1.81
...,...,...,...
210,Industrial_Zone,2025-11-01,6.76
212,Forest_Zone,2025-11-01,13.44
213,Industrial_Zone,2025-12-01,6.88
214,Water_Intake_Zone,2025-12-01,5.57


,t,recharge_mm
count,202,202.000000
mean,2022-12-19 21:44:33.267326,19.715792
min,2020-01-01 00:00:00,0.000000
25%,2021-07-01 00:00:00,6.662500
50%,2022-12-16 12:00:00,17.035000
75%,2024-07-01 00:00:00,25.920000
max,2025-12-01 00:00:00,75.510000
std,NaN,16.584000


<class 'pandas.DataFrame'>
Index: 202 entries, 0 to 215
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   zone         202 non-null    str           
 1   t            202 non-null    datetime64[us]
 2   recharge_mm  202 non-null    float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 9.2 KB


None

In [53]:
df_recharge.to_parquet(f"{data_binary_representation}recharge.parquet", engine="pyarrow")

In [122]:
df_sources = pd.read_csv(f"{original_data_path}sources.csv")

In [123]:
df_sources["start_date"] = pd.to_datetime(df_sources["start_date"], format="mixed", errors="coerce")
df_sources["end_date"] = pd.to_datetime(df_sources["end_date"], format="%d.%m.%Y", errors="coerce")
df_sources["end_date"] = df_sources["end_date"].fillna(pd.to_datetime("2026-01-01"))
df_sources.loc[df_sources["estimated_release_rate_g_day"] < 0, "estimated_release_rate_g_day"] = np.nan

df_sources["estimated_release_rate_g_day"] = df_sources["estimated_release_rate_g_day"].fillna(df_sources["estimated_release_rate_g_day"].mean())

In [124]:
display(df_sources)
display(df_sources.describe())
display(df_sources.info())

,source_id,name,x,y,z_top,z_bottom,start_date,end_date,estimated_release_rate_g_day
0,SRC_01,Цех обезжиривания №1,950.0,1020.0,-15.0,-45.0,2020-01-01,2026-01-01,450.0
1,SRC_02,Старый шламонакопитель,1120.5,940.0,-5.0,-30.0,2020-03-15,2023-12-31,320.0
2,SRC_03,Склад растворителей,1030.0,1100.0,-10.0,-25.0,2020-10-06,2026-01-01,180.0
3,SRC_04,Дренажный коллектор Юг,890.0,850.0,-20.0,-55.0,2020-02-01,2026-01-01,450.0
4,SRC_05,Аварийный сброс 2021,1450.0,1300.0,-8.0,-20.0,2021-12-08,2022-09-15,850.0


,x,y,z_top,z_bottom,start_date,end_date,estimated_release_rate_g_day
count,5.000000,5.000000,5.00000,5.00000,5,5,5.00000
mean,1088.100000,1042.000000,-11.60000,-35.00000,2020-08-06 04:48:00,2024-12-09 19:12:00,450.00000
min,890.000000,850.000000,-20.00000,-55.00000,2020-01-01 00:00:00,2022-09-15 00:00:00,180.00000
25%,950.000000,940.000000,-15.00000,-45.00000,2020-02-01 00:00:00,2023-12-31 00:00:00,320.00000
50%,1030.000000,1020.000000,-10.00000,-30.00000,2020-03-15 00:00:00,2026-01-01 00:00:00,450.00000
75%,1120.500000,1100.000000,-8.00000,-25.00000,2020-10-06 00:00:00,2026-01-01 00:00:00,450.00000
max,1450.000000,1300.000000,-5.00000,-20.00000,2021-12-08 00:00:00,2026-01-01 00:00:00,850.00000
std,220.063741,171.522593,5.94138,14.57738,NaN,NaN,249.89998


<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   source_id                     5 non-null      str           
 1   name                          5 non-null      str           
 2   x                             5 non-null      float64       
 3   y                             5 non-null      float64       
 4   z_top                         5 non-null      float64       
 5   z_bottom                      5 non-null      float64       
 6   start_date                    5 non-null      datetime64[us]
 7   end_date                      5 non-null      datetime64[us]
 8   estimated_release_rate_g_day  5 non-null      float64       
dtypes: datetime64[us](2), float64(5), str(2)
memory usage: 716.0 bytes


None

In [125]:
df_sources.to_parquet(f"{data_binary_representation}sources.parquet", engine="pyarrow")

In [126]:
permeability = tf.imread(f"{original_data_path}permeability.tif")

In [127]:
permeability[permeability == -9999.0] = np.nan

x, y = np.mgrid[0:permeability.shape[0], 0:permeability.shape[1]]

nan_mask = np.isnan(permeability)
valid_mask = ~np.isnan(permeability)

valid_coords = np.column_stack((x[valid_mask], y[valid_mask]))
valid_values = permeability[valid_mask]

nan_coords = np.column_stack((x[nan_mask], y[nan_mask]))
filled_values = griddata(valid_coords, valid_values, nan_coords, method='nearest')

permeability[nan_mask] = filled_values

In [128]:
main_df = pd.read_parquet(f"{data_binary_representation}wells.parquet")

index_x = np.clip((main_df["x"] // 10).astype(int), 0, 499)
index_y = np.clip((main_df["y"] // 10).astype(int), 0, 499)

main_df["permeability"] = permeability[index_x, index_y]

In [129]:
main_df.to_parquet(f"{data_binary_representation}wells.parquet", engine="pyarrow")